# KNN Economics – Reusable Template

**Short name:** `KNN_Econ`  
Drop in any numeric table with a binary macro / country / firm label. Scale on the training fold only.

```
load → split → MinMaxScaler.fit(train) → KNeighborsClassifier(k) → score / sweep k → simulate
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

# --- edit ---
CSV = "data/knn_econ_recession.csv"
FEATURE_COLS = None
LABEL = "recession"
TEST_SIZE = 0.2
SEED = 100
K_GRID = range(1, 31)
# ------------

df = pd.read_csv(CSV)
y = df[LABEL].to_numpy()
if FEATURE_COLS is None:
    X = df.drop(columns=[LABEL] + [c for c in df.columns if df[c].dtype == object]).to_numpy()
else:
    X = df[FEATURE_COLS].to_numpy()

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
sc = MinMaxScaler()
Xtr_s, Xva_s = sc.fit_transform(Xtr), sc.transform(Xva)

accs = []
for k in K_GRID:
    accs.append(KNeighborsClassifier(n_neighbors=k).fit(Xtr_s, ytr).score(Xva_s, yva))
best_k = list(K_GRID)[int(np.argmax(accs))]
print(f"n={len(df)} features={X.shape[1]} best_k={best_k} acc={max(accs):.4f}")
print("naive majority-class floor:", max(y.mean(), 1 - y.mean()))

clf = KNeighborsClassifier(n_neighbors=best_k).fit(Xtr_s, ytr)
print(classification_report(yva, clf.predict(Xva_s), digits=3))

plt.plot(list(K_GRID), accs)
plt.axvline(best_k, ls="--")
plt.xlabel("k"); plt.ylabel("valid acc"); plt.title("KNN sweep")
plt.show()
